# Portfolio Advisor Decision Intelligence (LangGraph + AgentCore Metadata)

## Introduction

This tutorial builds a **LangGraph-based wealth-management advisor** that demonstrates the single most impactful metadata pattern from the blog — **temporal filtering** — while going further with **recommendation-disposition tracking**.

Every portfolio discussion is tagged with what the client actually *did* with the advice: `accepted`, `rejected`, `deferred`, `modified`, or `no_recommendation_made`. Combined with market-regime inference (`bull`, `bear`, `volatile`, `sideways`) and AUM-tier scoping, the advisor can answer questions like *"what advice has this client rejected in volatile markets?"* — a learning-loop query that's impossible with semantic search alone.

The notebook also demonstrates **two strategies sharing the same indexed keys** — a `SemanticMemoryStrategy` capturing disposition and regime, and a `UserPreferenceMemoryStrategy` capturing the client's stated risk posture — with each strategy's own extraction instructions tuned for its purpose.

### Tutorial Details

| Information         | Details                                                                            |
|:--------------------|:-----------------------------------------------------------------------------------|
| Tutorial type       | Long-term memory with temporal + metadata filtering                                |
| Agent type          | Financial services portfolio advisor                                               |
| Agentic Framework   | LangGraph                                                                          |
| LLM model           | Anthropic Claude Haiku 4.5                                                         |
| Tutorial components | Two strategies with shared indexed keys, temporal + disposition + regime filters   |
| Example complexity  | Intermediate–Advanced                                                              |

### You'll learn to
- Define **two strategies sharing** the same four indexed keys (semantic + user-preference)
- Use **system-generated `x-amz-agentcore-memory-createdAt`** with `BEFORE` / `AFTER` for quarterly temporal brackets
- Track `recommendation_disposition` — the learning loop that turns advisor memory into advisor intelligence
- Use `NUMBER` filtering (`tax_impact_usd > 10000`) for material tax-relevant memories
- Use `update_memory` to add new indexed keys as the schema evolves
- Integrate with LangGraph's `create_react_agent` via pre/post-model hooks

## Prerequisites
- Python 3.10+
- AWS credentials with `bedrock-agentcore` and `bedrock-agentcore-control` permissions
- A `memory_execution_role_arn`
- Amazon Bedrock access to Anthropic Claude Haiku 4.5


## Step 1: Install Dependencies

In [ ]:
!pip install -qr requirements.txt

## Step 2: Imports and Configuration

In [ ]:
import os
import json
import uuid
import time
import logging
from datetime import datetime, timedelta, timezone
from typing import Optional, List, Dict

import boto3
from botocore.exceptions import ClientError

from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.runnables import RunnableConfig
from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import InMemorySaver

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger("portfolio-advisor-metadata")
logger.info("Imports loaded")


In [ ]:
# Replace with your values
REGION = "us-west-2"
MEMORY_EXECUTION_ROLE_ARN = "arn:aws:iam::<ACCOUNT_ID>:role/<AgentCoreMemoryExecutionRole>"
MODEL_ID = "global.anthropic.claude-haiku-4-5-20251001-v1:0"

CLIENT_ID = "client-789"
SESSION_ID = f"advisory_{datetime.now().strftime('%Y%m%d%H%M%S')}"

logger.info(f"Region:  {REGION}")
logger.info(f"Client:  {CLIENT_ID}")
logger.info(f"Session: {SESSION_ID}")


## Step 3: Create Memory with Two Strategies and Shared Indexed Keys

This is where the tutorial's distinctive structure lives: **one set of indexed keys shared across two memory strategies**.

- `SemanticMemoryStrategy` captures portfolio *facts* — what was discussed, what was recommended, what the client did.
- `UserPreferenceMemoryStrategy` captures the client's *stated preferences* — risk posture over time.

Both reference the same four indexed keys (`market_regime`, `recommendation_disposition`, `aum_tier`, `tax_impact_usd`), but each has its own `metadataSchema` with extraction instructions tuned to its purpose. This is the *"one index budget, multiple extraction personalities"* pattern the blog calls out as a design best practice.

The workhorse field is `recommendation_disposition`. Semantic search has no reliable signal for *"the client rejected this recommendation"* — that fact is usually implicit in a later turn. Making it a structured label turns it into a first-class filter.


In [ ]:
control_client = boto3.client("bedrock-agentcore-control", region_name=REGION)

memory_name = "AdvisorDecisionIntelligenceMemory"

indexed_keys = [
    {"key": "market_regime",              "type": "STRING"},
    {"key": "recommendation_disposition", "type": "STRING"},
    {"key": "aum_tier",                   "type": "STRING"},
    {"key": "tax_impact_usd",             "type": "NUMBER"},
]

# Schema used by both strategies — same keys, slightly different extraction emphasis.
def regime_schema(for_preference: bool):
    base_instruction = (
        "Classify the market regime based on conversation context and any explicit market "
        "commentary. bull = sustained uptrend, bear = sustained downtrend, sideways = range-bound, "
        "volatile = high realized vol regardless of direction."
    )
    if for_preference:
        base_instruction += " For preference records, set to the regime active when the client stated the preference."
    return {
        "key": "market_regime",
        "type": "STRING",
        "extractionConfig": {
            "llmExtractionConfig": {
                "definition": "Market regime during the discussion.",
                "llmExtractionInstruction": base_instruction,
                "validation": {
                    "stringValidation": {"allowedValues": ["bull", "bear", "sideways", "volatile"]}
                }
            }
        }
    }

disposition_schema_semantic = {
    "key": "recommendation_disposition",
    "type": "STRING",
    "extractionConfig": {
        "llmExtractionConfig": {
            "definition": (
                "What the client did with the recommendation offered in the conversation. "
                "accepted = agreed to execute; rejected = explicitly declined; "
                "deferred = postponed decision; modified = accepted with changes; "
                "no_recommendation_made = conversation did not include a concrete recommendation."
            ),
            "llmExtractionInstruction": (
                "Read the entire conversation and determine the client's disposition toward the "
                "advisor's recommendation. Default to no_recommendation_made if no concrete proposal was discussed."
            ),
            "validation": {
                "stringValidation": {
                    "allowedValues": ["accepted", "rejected", "deferred", "modified", "no_recommendation_made"]
                }
            }
        }
    }
}

disposition_schema_preference = {
    "key": "recommendation_disposition",
    "type": "STRING",
    "extractionConfig": {
        "llmExtractionConfig": {
            "definition": "Disposition the client expressed toward the advice at the time this preference was stated.",
            "llmExtractionInstruction": (
                "For preference records, infer disposition from language the client used when expressing "
                "the preference. Default to no_recommendation_made."
            ),
            "validation": {
                "stringValidation": {
                    "allowedValues": ["accepted", "rejected", "deferred", "modified", "no_recommendation_made"]
                }
            }
        }
    }
}

aum_schema = {
    "key": "aum_tier",
    "type": "STRING",
    "extractionConfig": {
        "llmExtractionConfig": {
            "definition": "Client AUM segment: under_100k, 100k_1m, 1m_10m, over_10m.",
            "llmExtractionInstruction": "LATEST_VALUE",
            "validation": {
                "stringValidation": {
                    "allowedValues": ["under_100k", "100k_1m", "1m_10m", "over_10m"]
                }
            }
        }
    }
}

tax_schema = {
    "key": "tax_impact_usd",
    "type": "NUMBER",
    "extractionConfig": {
        "llmExtractionConfig": {
            "definition": "USD tax implication of the discussed action (positive = tax cost, 0 = neutral).",
            "llmExtractionInstruction": (
                "Infer from any dollar figures + tax language in the conversation. If only a percentage "
                "is given, convert using any explicit portfolio value mentioned. Output a USD number."
            )
        }
    }
}

semantic_metadata_schema = [
    regime_schema(for_preference=False),
    disposition_schema_semantic,
    aum_schema,
    tax_schema,
]

preference_metadata_schema = [
    regime_schema(for_preference=True),
    disposition_schema_preference,
    aum_schema,
    tax_schema,
]

try:
    resp = control_client.create_memory(
        name=memory_name,
        eventExpiryDuration=180,
        memoryExecutionRoleArn=MEMORY_EXECUTION_ROLE_ARN,
        indexedKeys=indexed_keys,
        memoryStrategies=[
            {
                "semanticMemoryStrategy": {
                    "name": "PortfolioDiscussionSemantic",
                    "description": "Portfolio discussion facts with regime/disposition metadata",
                    "namespaces": ["wealth/{actorId}/facts/"],
                    "memoryRecordSchema": {"metadataSchema": semantic_metadata_schema}
                }
            },
            {
                "userPreferenceMemoryStrategy": {
                    "name": "ClientRiskPreferences",
                    "description": "Client investment preferences with shared indexed metadata",
                    "namespaces": ["wealth/{actorId}/preferences/"],
                    "memoryRecordSchema": {"metadataSchema": preference_metadata_schema}
                }
            }
        ],
        clientToken=str(uuid.uuid4()),
    )
    memory_id = resp["memory"]["id"]
    logger.info(f"✅ Created memory {memory_id}")
except ClientError as e:
    if e.response["Error"]["Code"] == "ConflictException":
        existing = control_client.list_memories()["memories"]
        match = next((m for m in existing if m["name"] == memory_name), None)
        memory_id = match["id"]
        logger.info(f"ℹ️  Reusing existing memory {memory_id}")
    else:
        raise

facts_ns = f"wealth/{CLIENT_ID}/facts/"
prefs_ns = f"wealth/{CLIENT_ID}/preferences/"
logger.info(f"Facts namespace:       {facts_ns}")
logger.info(f"Preferences namespace: {prefs_ns}")


## Step 4: Wait for Memory to be Active


In [ ]:
def wait_active(mem_id, timeout_s=240):
    deadline = time.time() + timeout_s
    while time.time() < deadline:
        m = control_client.get_memory(memoryId=mem_id)["memory"]
        logger.info(f"status: {m['status']}")
        if m["status"] == "ACTIVE":
            return
        if m["status"] == "FAILED":
            raise RuntimeError(m.get("failureReason"))
        time.sleep(10)
    raise TimeoutError("memory never reached ACTIVE")

wait_active(memory_id)


## Step 5: Seed Historical Portfolio Discussions

We backdate events across three simulated quarters so the system-generated `x-amz-agentcore-memory-createdAt` reflects the historical timeline. Each event carries `aum_tier` as event-time metadata (known from the client record); `market_regime`, `recommendation_disposition`, and `tax_impact_usd` are all FM-inferred from the conversation content.

The seeded set covers:
- Q1: a **rejected** rebalancing recommendation during a **bull** market.
- Q2: an **accepted** tax-loss-harvesting recommendation during a **volatile** market with ~$18K tax impact.
- Q3: a **deferred** international equity allocation during a **bear** market.
- Q4 (current): an **accepted** treasury allocation with modest tax impact.

This gives us meaningful corpus density to demonstrate disposition + regime filtering.


In [ ]:
data_client = boto3.client("bedrock-agentcore", region_name=REGION)

AUM = "1m_10m"

def submit(session_suffix, turns, backdate_days):
    session_id = f"advisory-{session_suffix}"
    base_ts = datetime.now(timezone.utc) - timedelta(days=backdate_days)
    payload = []
    for role, text in turns:
        payload.append({
            "conversational": {
                "role": "USER" if role == "user" else "ASSISTANT",
                "content": {"text": text},
            }
        })
    data_client.create_event(
        memoryId=memory_id,
        actorId=CLIENT_ID,
        sessionId=session_id,
        eventTimestamp=base_ts,
        payload=payload,
        metadata={"aum_tier": {"stringValue": AUM}},
    )
    logger.info(f"submitted {session_suffix} backdated {backdate_days}d")

# Q1 — rejected rebalance in a bull market
submit("q1-rejected-rebalance", [
    ("user",      "Markets have had another strong quarter. Should we rebalance?"),
    ("assistant", "Your equity allocation has drifted from 60% to 72%. I recommend rebalancing back to target, which would realize approximately $45,000 in short-term gains — call it ~$14,000 in taxes."),
    ("user",      "No — I don't want to pay the tax drag. Let it ride."),
    ("assistant", "Understood. We'll continue to monitor and revisit if the drift exceeds 15 percentage points."),
], backdate_days=240)

# Q2 — accepted tax-loss harvesting in volatile market
submit("q2-accepted-tlh", [
    ("user",      "The market is whiplashing. Anything we should do?"),
    ("assistant", "With the volatility, you have ~$60,000 of realized losses across your mid-cap positions. Recommend a tax-loss harvest to preserve ~$18,000 of tax alpha while maintaining factor exposure via a substantially-identical fund swap."),
    ("user",      "Yes, proceed. Execute."),
    ("assistant", "Executing the swap now. Wash-sale window noted."),
], backdate_days=150)

# Q3 — deferred international allocation in bear market
submit("q3-deferred-intl", [
    ("user",      "Markets keep falling. Does it make sense to add international exposure?"),
    ("assistant", "A 10% international allocation funded from US large-cap would improve diversification. Tax impact roughly $8,000. Recommended."),
    ("user",      "Let me think about it. I'll decide next quarter."),
    ("assistant", "Noted — deferred. I'll flag for revisit next quarter."),
], backdate_days=75)

# Q4 — accepted treasury allocation (current)
submit("q4-accepted-treasuries", [
    ("user",      "Should we lock in the high yields on the short end of the curve?"),
    ("assistant", "A 20% allocation to 1-year T-bills yields ~5% with negligible tax drag — perhaps $2,000. Recommended."),
    ("user",      "Do it."),
    ("assistant", "Allocation executed."),
], backdate_days=5)

# Preference setting — client states conservative posture under bear market
submit("pref-conservative-bear", [
    ("user",      "For the record, whenever markets are in a clear bear, I prefer a defensive tilt — cash and treasuries over risk assets."),
    ("assistant", "Noted — documenting your preference for a defensive posture during bear regimes."),
], backdate_days=75)

logger.info("Seeded 5 sessions across 3 quarters + 1 preference")


## Step 6: Wait for Extraction Across Both Strategies


In [ ]:
def wait_records(ns, expected_min=1, timeout_s=240):
    deadline = time.time() + timeout_s
    while time.time() < deadline:
        r = data_client.list_memory_records(memoryId=memory_id, namespace=ns, maxResults=50)
        recs = r.get("memoryRecordSummaries", [])
        logger.info(f"[{ns}] records so far: {len(recs)}")
        if len(recs) >= expected_min:
            return recs
        time.sleep(15)
    raise TimeoutError(f"only {len(recs)} in {ns}")

facts_records = wait_records(facts_ns, expected_min=2)
pref_records = wait_records(prefs_ns, expected_min=1)

logger.info(f"✅ semantic={len(facts_records)}  user_preference={len(pref_records)}")


## Step 7: Inspect Extracted Metadata Across Both Strategies

The same four indexed keys appear on records from **both** namespaces — but the values reflect each strategy's extraction focus. Semantic records show the disposition of *each specific discussion*; preference records show the disposition attached to the *stated preference*.


In [ ]:
def dump(ns, label):
    recs = data_client.list_memory_records(memoryId=memory_id, namespace=ns, maxResults=20).get("memoryRecordSummaries", [])
    print(f"\n### {label} ({len(recs)} records)")
    for r in recs:
        full = data_client.get_memory_record(memoryId=memory_id, memoryRecordId=r["memoryRecordId"])["memoryRecord"]
        text = full.get("content", {}).get("text", "")[:150]
        md_entries = full.get("metadata", {})
        print(f"\n  content: {text}")
        for k, v in md_entries.items():
            print(f"    {k}: {v}")

dump(facts_ns, "SEMANTIC (facts)")
dump(prefs_ns, "USER_PREFERENCE (preferences)")


## Step 8: Temporal Filtering — the Q-Bracket Query

The blog's strongest metadata claim: *"LLMs are notoriously poor at parsing temporal constraints from natural language. Structured DATETIME filtering with BEFORE and AFTER operators convert this into a deterministic index lookup."*

Bracket on `x-amz-agentcore-memory-createdAt` with `AFTER <90d-ago>` and `BEFORE <30d-ago>` to get Q3 only — ignoring Q1, Q2, and Q4.


In [ ]:
def retrieve(ns, query, filters=None, top_k=10):
    s = {"searchQuery": query, "topK": top_k}
    if filters:
        s["metadataFilters"] = filters
    r = data_client.retrieve_memory_records(memoryId=memory_id, namespace=ns, searchCriteria=s)
    return r.get("memoryRecordSummaries", [])

def show(results, label):
    print(f"\n--- {label}: {len(results)} results ---")
    for r in results:
        print(f"  [{r.get('score', 0):.3f}] {r.get('content', {}).get('text', '')[:180]}")

q3_after  = (datetime.now(timezone.utc) - timedelta(days=90)).strftime("%Y-%m-%dT%H:%M:%SZ")
q3_before = (datetime.now(timezone.utc) - timedelta(days=30)).strftime("%Y-%m-%dT%H:%M:%SZ")

q3_only = retrieve(
    facts_ns, "rebalancing strategy",
    filters=[
        {"left": {"metadataKey": "x-amz-agentcore-memory-createdAt"}, "operator": "AFTER",
         "right": {"metadataValue": {"dateTimeValue": q3_after}}},
        {"left": {"metadataKey": "x-amz-agentcore-memory-createdAt"}, "operator": "BEFORE",
         "right": {"metadataValue": {"dateTimeValue": q3_before}}},
    ],
)
show(q3_only, f"TEMPORAL: createdAt AFTER {q3_after} AND BEFORE {q3_before}")


## Step 9: The Learning-Loop Query — *"What has this client rejected?"*

This is the query that's impossible with semantic search alone. Filter by `recommendation_disposition = rejected` across the full client history.


In [ ]:
rejected = retrieve(
    facts_ns, "advice recommendation",
    filters=[
        {"left": {"metadataKey": "recommendation_disposition"}, "operator": "EQUALS_TO",
         "right": {"metadataValue": {"stringValue": "rejected"}}},
    ],
)
show(rejected, "disposition = rejected")


## Step 10: Regime-Aware Disposition

*"Recommendations this client rejected during volatile markets."* — combines two STRING filters. Useful for observing patterns: does the client reject differently in different regimes?


In [ ]:
rejected_volatile = retrieve(
    facts_ns, "advice recommendation",
    filters=[
        {"left": {"metadataKey": "recommendation_disposition"}, "operator": "EQUALS_TO",
         "right": {"metadataValue": {"stringValue": "rejected"}}},
        {"left": {"metadataKey": "market_regime"}, "operator": "EQUALS_TO",
         "right": {"metadataValue": {"stringValue": "volatile"}}},
    ],
)
show(rejected_volatile, "rejected + volatile")


## Step 11: Material Tax Events

`tax_impact_usd > 10000` surfaces only materially tax-relevant memories.


In [ ]:
material_tax = retrieve(
    facts_ns, "tax",
    filters=[
        {"left": {"metadataKey": "tax_impact_usd"}, "operator": "GREATER_THAN",
         "right": {"metadataValue": {"numberValue": 10000}}},
    ],
)
show(material_tax, "tax_impact_usd > $10,000")


## Step 12: Cross-Strategy Retrieval

Pull *facts* from the semantic namespace filtered by regime AND *preferences* from the preference namespace filtered by the same regime. Both share `market_regime` as an indexed key, so the filter works identically across strategies.


In [ ]:
bear_facts = retrieve(
    facts_ns, "defensive",
    filters=[
        {"left": {"metadataKey": "market_regime"}, "operator": "EQUALS_TO",
         "right": {"metadataValue": {"stringValue": "bear"}}},
    ],
)
bear_prefs = retrieve(
    prefs_ns, "defensive",
    filters=[
        {"left": {"metadataKey": "market_regime"}, "operator": "EQUALS_TO",
         "right": {"metadataValue": {"stringValue": "bear"}}},
    ],
)
show(bear_facts, "FACTS (semantic) during bear regime")
show(bear_prefs, "PREFERENCES during bear regime")


## Step 13: Non-Semantic Enumeration

Sometimes the cleanest way to answer *"what has this client said no to?"* is a direct `list_memory_records` with no semantic search involved at all.


In [ ]:
audit = data_client.list_memory_records(
    memoryId=memory_id,
    namespace=facts_ns,
    metadataFilters=[
        {"left": {"metadataKey": "recommendation_disposition"}, "operator": "EQUALS_TO",
         "right": {"metadataValue": {"stringValue": "rejected"}}},
    ],
)
recs = audit.get("memoryRecordSummaries", [])
print(f"All rejected recommendations ({len(recs)}):")
for r in recs:
    print(f"  - {r.get('content', {}).get('text', '')[:200]}")


## Step 14: LangGraph Agent with Metadata-Filtered Retrieval

Wire the filtered retrieval into a LangGraph react agent via a tool. The agent can use filters (disposition, regime, quarter-range) to retrieve the exact slice of history that shapes the current answer.


In [ ]:
def make_retrieval_tool():
    from langchain_core.tools import tool

    @tool
    def retrieve_prior_advice(
        query: str,
        disposition: Optional[str] = None,
        regime: Optional[str] = None,
        min_days_ago: Optional[int] = None,
        max_days_ago: Optional[int] = None,
    ) -> str:
        """Retrieve prior portfolio discussions from this client's memory, filtered by
        disposition, market regime, and/or a time window.

        Args:
            query: free-text query for semantic relevance within the filtered set.
            disposition: optional, one of accepted, rejected, deferred, modified, no_recommendation_made.
            regime: optional, one of bull, bear, sideways, volatile.
            min_days_ago: optional lower bound on recency (excludes records older than N days).
            max_days_ago: optional upper bound on recency (excludes records newer than N days).
        """
        filters = []
        if disposition:
            filters.append({"left": {"metadataKey": "recommendation_disposition"}, "operator": "EQUALS_TO",
                            "right": {"metadataValue": {"stringValue": disposition}}})
        if regime:
            filters.append({"left": {"metadataKey": "market_regime"}, "operator": "EQUALS_TO",
                            "right": {"metadataValue": {"stringValue": regime}}})
        if min_days_ago is not None:
            cutoff = (datetime.now(timezone.utc) - timedelta(days=min_days_ago)).strftime("%Y-%m-%dT%H:%M:%SZ")
            filters.append({"left": {"metadataKey": "x-amz-agentcore-memory-createdAt"}, "operator": "BEFORE",
                            "right": {"metadataValue": {"dateTimeValue": cutoff}}})
        if max_days_ago is not None:
            cutoff = (datetime.now(timezone.utc) - timedelta(days=max_days_ago)).strftime("%Y-%m-%dT%H:%M:%SZ")
            filters.append({"left": {"metadataKey": "x-amz-agentcore-memory-createdAt"}, "operator": "AFTER",
                            "right": {"metadataValue": {"dateTimeValue": cutoff}}})
        recs = retrieve(facts_ns, query, filters=filters or None, top_k=5)
        if not recs:
            return "no matching prior advice"
        return "\n".join(f"- {r.get('content', {}).get('text', '')[:220]}" for r in recs)

    return retrieve_prior_advice

retrieve_tool = make_retrieval_tool()

llm = init_chat_model(MODEL_ID, model_provider="bedrock_converse", region_name=REGION)

agent = create_react_agent(
    llm,
    tools=[retrieve_tool],
    checkpointer=InMemorySaver(),
)

config = {"configurable": {"thread_id": SESSION_ID}}
question = "Given this client's history, how should I frame another rebalancing recommendation?"
print("QUESTION:", question, "\n")
result = agent.invoke({"messages": [HumanMessage(content=question)]}, config)
print("ANSWER:\n", result["messages"][-1].content)


## Step 15: Additive Schema Evolution

Add a new indexed key mid-lifetime. Existing records don't retroactively receive the new field, but they acquire it as they're consolidated with newer ones. Removing an indexed key is intentionally disallowed.


In [ ]:
try:
    control_client.update_memory(
        memoryId=memory_id,
        addIndexedMetadataKeys=[
            {"metadataKey": "instrument_type", "metadataValueType": "STRING"}
        ],
    )
    logger.info("✅ Added instrument_type as a new indexed key")
except ClientError as e:
    logger.warning(f"update_memory: {e}")

updated = control_client.get_memory(memoryId=memory_id)["memory"]
print("Indexed keys now:")
for k in updated.get("indexedKeys", []):
    print(f"  - {k['key']} ({k['type']})")


## Step 16: Cleanup (Optional)


In [ ]:
# control_client.delete_memory(memoryId=memory_id)
# print(f"Deleted {memory_id}")


## What you built

- **Two strategies sharing one set of indexed keys** — semantic (facts) + user-preference (client posture). Each strategy has its own `metadataSchema` with extraction instructions tuned for its purpose.
- **Temporal filtering** via the system-generated `x-amz-agentcore-memory-createdAt` — the canonical Q-bracket query using `AFTER` + `BEFORE` without declaring any datetime indexed key of your own.
- **Recommendation-disposition tracking** — the learning-loop query that lets the advisor learn from what the client actually did, not just what was discussed.
- **Cross-strategy retrieval** — the same `market_regime` filter applied across both namespaces, returning different record *types* scoped by the same business dimension.
- **NUMBER filter** — `tax_impact_usd > 10000` to surface only materially tax-relevant memories.
- **`update_memory`** schema evolution — additive-only addition of new indexed keys.
- LangGraph agent integration via a filtered-retrieval tool.

### Takeaways
- Structured metadata is what turns advisor memory into **advisor intelligence** — learning loops, regime awareness, and temporal precision that semantic search can't express.
- Two strategies can share a single indexed-key budget, each populating the shared keys differently based on its extraction focus.
- Temporal filtering on `x-amz-agentcore-memory-createdAt` is the single highest-leverage metadata pattern for any time-sensitive domain.
